# 199. RoPE 长上下文：Position Interpolation 与缩放合同怎样实现？

> **面试问题：RoPE 的相对位置性质是什么？怎样缩放位置、验证短/长上下文、并防止不同 RoPE 配置的 KV cache 错误复用？**

## 先给结论

面试中不能只背术语；需要把数学坐标、消息状态、协议顺序或模板字节流变成可检验的状态机。下列代码只使用标准库和小数组，明确教学 oracle 与生产替换点；它们不等同于真实模型效果、网络可靠性或正式安全认证。

## 一手资料

- [RoFormer / RoPE](https://arxiv.org/abs/2104.09864)
- [Position Interpolation](https://arxiv.org/abs/2306.15595)
- [YaRN](https://arxiv.org/abs/2309.00071)

In [ ]:
notebook_contract = {"mode": "in-memory-demo", "oracle": "explicit-assertions", "production": "versioned-and-observed"}  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["mode"] == "in-memory-demo"  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["oracle"] == "explicit-assertions"  # 执行本行的状态、计算或校验逻辑。
assert "observed" in notebook_contract["production"]  # 执行本行的状态、计算或校验逻辑。
assert len(notebook_contract) == 3  # 执行本行的状态、计算或校验逻辑。


## 1. 问题拆解：RoPE 将位置编码为二维旋转

RoPE 不在 token embedding 上简单相加位置向量，而是在每对通道上按位置旋转 Q/K。旋转保持向量范数，并使两个位置的内积只依赖相对位移；这正是长上下文缩放必须谨慎处理的坐标合同。


In [ ]:
import math  # 执行本行的状态、计算或校验逻辑。
def rotate_pair(vector, position, theta=0.1):  # 执行本行的状态、计算或校验逻辑。
    x0, x1 = vector  # 执行本行的状态、计算或校验逻辑。
    angle = position * theta  # 执行本行的状态、计算或校验逻辑。
    return (x0 * math.cos(angle) - x1 * math.sin(angle), x0 * math.sin(angle) + x1 * math.cos(angle))  # 执行本行的状态、计算或校验逻辑。
base_vector = (3.0, 4.0)  # 执行本行的状态、计算或校验逻辑。
at_zero = rotate_pair(base_vector, 0)  # 执行本行的状态、计算或校验逻辑。
assert at_zero == base_vector  # 执行本行的状态、计算或校验逻辑。
assert math.isclose(sum(value * value for value in rotate_pair(base_vector, 7)), 25.0)  # 执行本行的状态、计算或校验逻辑。
assert rotate_pair(base_vector, 1) != base_vector  # 执行本行的状态、计算或校验逻辑。


## 2. 相对位置性质：同时旋转 Q/K 等价于旋转相对位移

注意力分数的关键 oracle 是 `dot(R(p)q, R(t)k) = dot(q, R(t-p)k)`。代码直接比较两侧浮点值；生产 kernel 会向量化到所有 head/dim，仍应保留这种数值回归测试。


In [ ]:
def dot(left, right):  # 执行本行的状态、计算或校验逻辑。
    return sum(a * b for a, b in zip(left, right))  # 执行本行的状态、计算或校验逻辑。
q = (1.0, 2.0)  # 执行本行的状态、计算或校验逻辑。
k = (2.0, -1.0)  # 执行本行的状态、计算或校验逻辑。
left = dot(rotate_pair(q, 11), rotate_pair(k, 17))  # 执行本行的状态、计算或校验逻辑。
right = dot(q, rotate_pair(k, 6))  # 执行本行的状态、计算或校验逻辑。
assert math.isclose(left, right, rel_tol=1e-9)  # 执行本行的状态、计算或校验逻辑。
assert math.isclose(dot(rotate_pair(q, 5), rotate_pair(k, 5)), dot(q, k), rel_tol=1e-9)  # 执行本行的状态、计算或校验逻辑。
assert not math.isclose(left, dot(q, k), rel_tol=1e-3)  # 执行本行的状态、计算或校验逻辑。


## 3. Position Interpolation：把长位置压回训练位置范围

PI 的最小接口是把推理位置除以扩展因子。它不是免费外推：因果顺序仍用原始 token 顺序，只有旋转坐标被缩放；训练、KV cache、模板和评测长度也都需要与缩放配置绑定。


In [ ]:
def interpolate_position(position, factor):  # 执行本行的状态、计算或校验逻辑。
    if factor < 1:  # 执行本行的状态、计算或校验逻辑。
        raise ValueError("扩展因子不能小于一")  # 执行本行的状态、计算或校验逻辑。
    return position / factor  # 执行本行的状态、计算或校验逻辑。
assert interpolate_position(8192, 4) == 2048.0  # 执行本行的状态、计算或校验逻辑。
assert interpolate_position(0, 8) == 0.0  # 执行本行的状态、计算或校验逻辑。
assert interpolate_position(4096, 2) < 4096  # 执行本行的状态、计算或校验逻辑。


## 4. 频率：不同维度不能使用同一个旋转速度

真实 RoPE 对每个二维通道使用不同频率。下面显式产生频率表；低频通道承载更长尺度的信息，高频通道对局部位置更敏感。改 base 或缩放方案时，必须同时记录维度和频率公式。


In [ ]:
def rope_angles(position, dimension, base=10000.0):  # 执行本行的状态、计算或校验逻辑。
    if dimension % 2 != 0:  # 执行本行的状态、计算或校验逻辑。
        raise ValueError("RoPE 维度必须成对")  # 执行本行的状态、计算或校验逻辑。
    return [position / (base ** (2 * index / dimension)) for index in range(dimension // 2)]  # 执行本行的状态、计算或校验逻辑。
angles = rope_angles(100, 8)  # 执行本行的状态、计算或校验逻辑。
assert len(angles) == 4  # 执行本行的状态、计算或校验逻辑。
assert angles[0] > angles[-1]  # 执行本行的状态、计算或校验逻辑。
assert rope_angles(0, 8) == [0.0, 0.0, 0.0, 0.0]  # 执行本行的状态、计算或校验逻辑。


## 5. 缩放配置：训练长度、目标长度和方法共同组成契约

不能只在部署配置里把 maximum length 调大。是否微调、PI/YaRN 参数、短上下文回归集和 tokenizer/chat template 都会影响效果；这里先检查最基本的长度边界与方法白名单。


In [ ]:
def validate_scaling_config(train_length, target_length, method, factor):  # 执行本行的状态、计算或校验逻辑。
    return train_length > 0 and target_length >= train_length and factor >= 1 and method in {"none", "pi", "yarn"}  # 执行本行的状态、计算或校验逻辑。
config_ok = validate_scaling_config(2048, 8192, "pi", 4)  # 执行本行的状态、计算或校验逻辑。
assert config_ok is True  # 执行本行的状态、计算或校验逻辑。
assert not validate_scaling_config(2048, 1024, "pi", 1)  # 执行本行的状态、计算或校验逻辑。
assert not validate_scaling_config(2048, 8192, "unknown", 4)  # 执行本行的状态、计算或校验逻辑。


## 6. KV cache：缓存键必须携带绝对位置和 RoPE 版本

缩放不会改变 token 的因果顺序，但会改变 K 的旋转坐标。复用旧 KV 前必须比较 RoPE 配置和位置范围；否则同一 token id 的 key 向量可能来自不同的坐标系统。


In [ ]:
def kv_key(token_id, absolute_position, rope_version):  # 执行本行的状态、计算或校验逻辑。
    return (token_id, absolute_position, rope_version)  # 执行本行的状态、计算或校验逻辑。
cached = kv_key(42, 8192, "pi-4x-v1")  # 执行本行的状态、计算或校验逻辑。
assert cached == (42, 8192, "pi-4x-v1")  # 执行本行的状态、计算或校验逻辑。
assert cached != kv_key(42, 8192, "none-v1")  # 执行本行的状态、计算或校验逻辑。
assert cached != kv_key(42, 8193, "pi-4x-v1")  # 执行本行的状态、计算或校验逻辑。


## 7. 评测：短窗口回归与长距离任务必须并列

只测长上下文 needle 可能掩盖短窗口退化。最小报告应按原训练长度以内、长度外、不同证据位置和任务类型分桶；这里通过覆盖率表确保结果不会被单一平均数吞没。


In [ ]:
def summarize_eval(rows):  # 执行本行的状态、计算或校验逻辑。
    buckets = {"short": [], "long": []}  # 执行本行的状态、计算或校验逻辑。
    for row in rows:  # 执行本行的状态、计算或校验逻辑。
        buckets["short" if row["length"] <= 2048 else "long"].append(row["correct"])  # 执行本行的状态、计算或校验逻辑。
    return {name: sum(values) / len(values) for name, values in buckets.items() if values}  # 执行本行的状态、计算或校验逻辑。
report = summarize_eval([{"length": 512, "correct": 1}, {"length": 4096, "correct": 1}, {"length": 8192, "correct": 0}])  # 执行本行的状态、计算或校验逻辑。
assert report["short"] == 1.0  # 执行本行的状态、计算或校验逻辑。
assert report["long"] == 0.5  # 执行本行的状态、计算或校验逻辑。
assert set(report) == {"short", "long"}  # 执行本行的状态、计算或校验逻辑。


## 8. 制品：RoPE 参数需要同 checkpoint、KV 和评测关联

checkpoint 名称不足以标识位置系统。保存 base、维度、训练/目标长度、方法、factor 与 tokenizer/chat-template 版本，才能解释线上长度退化或 cache 不命中。


In [ ]:
import hashlib  # 执行本行的状态、计算或校验逻辑。
import json  # 执行本行的状态、计算或校验逻辑。
artifact = {"rope": "pi", "factor": 4, "train_length": 2048, "target_length": 8192, "dimension": 8}  # 执行本行的状态、计算或校验逻辑。
fingerprint = hashlib.sha256(json.dumps(artifact, sort_keys=True).encode()).hexdigest()  # 执行本行的状态、计算或校验逻辑。
assert artifact["rope"] == "pi"  # 执行本行的状态、计算或校验逻辑。
assert artifact["target_length"] > artifact["train_length"]  # 执行本行的状态、计算或校验逻辑。
assert len(fingerprint) == 64  # 执行本行的状态、计算或校验逻辑。


## 面试收束

回答时先说明不变量，再给出主路径和失败分支，最后说明指标、版本制品与生产替换点。不要把一个受控样例的通过误报成模型质量、可靠网络或端到端安全保证。
